# 1. Methodology: Taxonomic Harmonization and Cross-Batch Integration

Taxonomic Standardization and Structural Restructuring
To enable robust cross-batch integration and statistical comparison across multi-year sequencing projects, a standardized taxonomic nomenclature was enforced. Raw taxonomic lineages were systematically cleaned to remove low-confidence annotations beyond the genus level, discarding all species-level designations. Standard nomenclature formats were applied uniformly across all datasets using consistent capitalization and joining multi-word terms with underscores.

To optimize algorithmic sorting and indexing, the conventional hierarchical prefix notation was inverted. Lineages failing to resolve to lower taxonomic ranks were refactored from a prefix-dependent format (e.g., unclassified_FamilyName) to a suffix-dependent format (e.g., FamilyName_unclassified). This restructuring allowed for systematic, alphabetical grouping and matrix sorting strictly by hierarchy, moving sequentially from Kingdom down through Phylum, Class, Order, and Family. Missing phylum-level assignments were subsequently harmonized and remapped to modern phylum designations.
__Why GTDB Harmonisation__

GTDB goes a step beyond NCBI because it addresses a fundamental problem in classical taxonomy: polyphyly (groupings that don't share an exclusive common ancestor) and uneven evolutionary depth. Some incompatible results were looked in https://gtdb.ecogenomic.org
Standardized Evolutionary Distance: GTDB uses Relative Evolutionary Divergence (RED) based on 120 universal protein markers. This ensures that a "Phylum" or "Class" in GTDB represents the same relative amount of evolutionary change across the entire bacterial tree.
Resolution for Uncultivated Microbes: NCBI relies heavily on formal species descriptions (which require culturing). GTDB seamlessly incorporates metagenome-assembled genomes (MAGs) and single-cell genomes (SAGs), which make up over 80% of environmental microbial diversity.
Clarithy on Candidate Phyla: Uncultivated lineages like TM6 (Babelota) or Moranbacteria (Moraniibacteriota) have clean, normalized places in GTDB, whereas in NCBI they often linger in temporary bins like "candidate division."
Using NCBI / SILVA
Short-Read 16S Amplicon Sequencing:
Analyzing short 16S rRNA gene fragments (e.g., V4 or V3-V4 regions using QIIME 2 or Mothur), the SILVA or Greengenes2 reference databases are still standard. GTDB is designed primarily for genome/metagenome data (GTDB-Tk), though GTDB-mapped 16S databases do exist.
Formal Publications & Submissions: 
For submitting data to NCBI repositories (GenBank, SRA) or publishing in journals that strictly require LPSN (List of Prokaryotic names with Standing in Nomenclature) compliance,  will need to map back to validly published NCBI names. 

Unclear entries were looked directly in  https://gtdb.ecogenomic.org/    traditional family names like Comamonadaceae, Alcaligenaceae, and Sutterellaceae no longer exist as distinct families. When GTDB normalized relative evolutionary divergence (RED) across the genome tree, it determined that those groups were not distinct enough at the family rank to remain separate, so GTDB merged them into Burkholderiaceae.

Additionally, GTDB merged the entire class Betaproteobacteria into Gammaproteobacteria to resolve polyphyly in the Proteobacteria/Pseudomonadota lineage. Demonstrates the concatenated 120-marker protein phylogeny (bac120), introduces rank normalization via RED, and explicitly reclassifies Betaproteobacteria as a subgroup/heterotypic synonym nested within Gammaproteobacteri Parks et al. (2018) Details the standardized domain-to-species framework, ANI-based species cluster assignments, and systematic merging/curation of polyphyletic families across all bacterial lineages Parks et al. (2020).

__A Priori Baseline Selection and Environmental Filtering__
Cross-batch harmonization was guided by a deeply curated, high-confidence baseline taxonomy dataset comprising approximately 800 recognized genera. This baseline dataset, established via rigorous biological curation, was designated as the target structural framework. Given that samples spanned disparate computational workflows, laboratories, and collection timelines, the core microbial communities were expected to mirror the specialized ecological niches of the engineering systems under study.

To neutralize batch effects and eliminate false diversity expansions driven by technical artifacts or laboratory-specific misclassifications, unmapped or newly introduced taxa were managed using a conservative ecological filtering protocol. Unidentified or newly introduced operational taxonomic units (OTUs) were systematically collapsed into known baseline genera under the following restrictive conditions:

The target organism shared a direct, identical Family assignment with an established baseline representative.

The biological and ecological traits of the candidate genus aligned strictly with the specific environmental profiles of closed-loop industrial heating and cooling water infrastructure.

Taxa whose primary metadata tied them exclusively to disparate environmental systems—such as marine, deep-ocean, or petroleum reservoirs—were classified as cross-batch noise and omitted from direct genus-level mapping.

__Abundance Thresholding, Merging, and Mass Conservation__
When the incoming sequencing matrices introduced distinct taxonomic splits (e.g., separating an established baseline genus into multiple newly resolved variants), data integrity was maintained via abundance pooling. The abundance counts of these newly split taxa were collapsed back into the dominant, high-confidence baseline representative by summing their respective cell values, ensuring total absolute mass conservation within the sample profiles.

To protect the dataset from inflation by minor sequencing artifacts while retaining significant biological signals, unclassified lineages at the family level were preserved as valid analytical placeholders only if their individual relative abundance exceeded an ecological significance threshold of greater than 0.01%. High-abundance unclassified fractions critical to specific matrix profiles (such as the dominant unspecific_Bacteria_meta2 faction, representing 66% of Sample 1 total reads) were strictly locked and retained to prevent mathematical distortion of the remaining relative abundance profiles. All manual fusions, data corrections, and abundance shifts were permanently color-flagged within the master matrix to ensure complete traceability. Orange for the Genus representing the 2 or more genus and pink for the cell which get absorbed by the orange. Lila for the genus likely to become the representing genus. 

                     MIC PROJECT  
                        │  
          ┌─────────────┴─────────────┐  
          │                           │  
    Biotot_noncured ~800 GENERA    
   (preserved, never modified)       NEW 150/300 GENERA  
          │                           │  
          └─────────────┬─────────────┘  
                        │    
                    MultiTax   
                        │  
                        ▼  
                    GTDB R232  
                        │  
                        ▼  
                Current taxonomy 800-genus harmonised list  
                        │  
            ┌───────────┴───────────┐  
            │                       │  
    Update old reference     Classify new data  
            │                       │  
            ▼                       ▼  
    GID identity preserved       Match / new / unknown  



# 2. Importing packages and Directing paths

In [ ]:
from pathlib import Path
import pandas as pd
from multitax import GtdbTx
import inspect
import re
import openpyxl

In [3]:
gtdb = GtdbTx(version="232")
print(inspect.signature(GtdbTx))

URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

In [ ]:
# Load  current harmonised 800-genus reference
biotot_path = Path("data/Biotot.xlsx")
output_path = Path("data")
biotot_total = pd.read_excel(biotot_path, sheet_name="Biotot", engine = "openpyxl")
biotot = biotot_total[["Domain", "Phylum", "Class", "Order", "Family", "Genus",  "GID", "sum_all_sites"]]
biotot["GID"] = biotot["GID"].astype("Int16")
biotot["sum_all_sites"] = biotot["sum_all_sites"].astype("float32")

# print(biotot.head())

In [ ]:
def clean_genus_string(genus):
    """General cleanup: trim whitespace, collapse repeated/mixed
    whitespace and underscores into single underscores, strip
    leading/trailing underscores."""
    if pd.isna(genus):
        return genus
    genus = str(genus).strip()
    genus = re.sub(r"\s+", "_", genus)      # any whitespace -> underscore
    genus = re.sub(r"_+", "_", genus)       # collapse multiple underscores into one
    genus = genus.strip("_")                # drop leading/trailing underscores
    return genus

biotot["Genus"] = biotot["Genus"].apply(clean_genus_string)

# 3. Query GTDB R232 with MultiTax

Create a gtdb_taxonomy dataframe.
Query data from the GTDB database, to get the taxonomic ranks for each genus

In [ ]:
genus_queries = biotot[["Genus", "GID", "Family"]].copy()

In [ ]:
# Query data from the GTDB database, to get the taxonomic ranks for each genus
def clean_gtdb_query(genus):
    """
    Convert the project's Genus label into the best GTDB query name,
    while leaving the original Genus column untouched.
    """
    if pd.isna(genus):
        return None

    genus = str(genus).strip()
    prev = None
    while prev != genus:
        prev = genus
    if genus.startswith("Candidatus_"):
        genus = genus[len("Candidatus_"):]

    genus = re.sub(r"_[A-Za-z0-9-]+_group$", "", genus)
    genus = re.sub(r"_group$", "", genus)

    # Remove known placeholder suffixes
    genus = re.sub(
        r"_(unclassified|uncultured|unknown|bacterium|proteobacterium|group|sensu_stricto|clade)$", "", genus, flags=re.IGNORECASE)

    # Remove strain/sequence-style numeric suffix:
    # Selenomonas_3 -> Selenomonas
    genus = re.sub(r"_\d+$", "", genus)
    parts = genus.split("_")
    if len(parts) == 2 and parts[0][:1].isupper() and parts[1][:1].islower():
        genus = parts[0]
    
    return genus


In [ ]:
genus_queries["stripped"] = genus_queries["Genus"].apply(clean_gtdb_query)

# anything that stripped to empty string or just "Candidatus" alone = broken
suspect = genus_queries[genus_queries["stripped"].str.len() < 4]
suspect[["Genus", "stripped"]]

,Genus,stripped
68,K82,K82
533,A17,A17
616,OM1_unclassified,OM1
630,OM1_unclassified,OM1
644,TM6_uncultured,TM6


## 3.1 Double queryg Family and Genus

In [ ]:
def get_query_rank(original_genus):
    '''ranking it so that it queries depending on the rank of the genus, if it is a family or genus'''
    if pd.isna(original_genus):
        return None

    original_genus = str(original_genus).strip()

    # Family-level unresolved names # Family-level placeholder:# Rhodocyclaceae_unclassified -> Rhodocyclaceae
    # This will then be queried as a family, not as a genus.
    if original_genus.endswith("aceae"):
        return "f__"
    if original_genus.endswith("ales"):
        return "o__"
    if original_genus.endswith("_unclassified") or original_genus.endswith("_uncultured"):
        base = re.sub(
            r"_(unclassified|uncultured)$","", original_genus,
            flags=re.IGNORECASE
        )

        if base.endswith("aceae"):
            return "f__"

    # Numeric variants
    if re.search(r"_\d+$", original_genus):
        return "g__"

    # Normal genus
    return "g__"

In [ ]:
# the querry will have the Genus and keep the ID column, so to not lose the ID, id already in df
genus_queries["GTDB_query_genus"] = (genus_queries["Genus"].astype(str).map(clean_gtdb_query))

genus_queries["GTDB_query_rank"] = (genus_queries["Genus"].map(get_query_rank))

genus_queries["GTDB_query"] = (genus_queries["GTDB_query_rank"] + genus_queries["GTDB_query_genus"])

genus_queries["Query_transformation"] = (genus_queries["Genus"] != genus_queries["GTDB_query_genus"])


## 3.2 Lookup

In [ ]:
rank_prefix = {"d__": "Kingdom", "p__": "Phylum", "c__": "Class", "o__": "Order", "f__": "Family", "g__": "Genus"}

def lookup_lineage(gtdb_query):
    """Takes a full query string (e.g. 'g__Escherichia' or 'f__Aerococcaceae').
    Returns a dict of rank -> value if a real lineage was found, else None."""
    try:
        lineage_nodes = gtdb.lineage(gtdb_query)
    except Exception:
        return None
    parsed = {}
    for node in lineage_nodes:
        for prefix, rankname in rank_prefix.items():
            if node.startswith(prefix):
                parsed[rankname] = node[len(prefix):]
                break
    return parsed if parsed else None
def search_gtdb_nodes(substring):
    """Case-insensitive substring search across all GTDB node keys.
    Used to find suffixed variants (Calothrix_A) or check whether
    something exists under a different name."""
    substring = str(substring).lower()
    return [n for n in gtdb._nodes.keys() if substring in n.lower()]

def lookup_lineage_with_suffix_retry(base_name, rank_prefix_char="g__"):
    """Try exact match first. If that fails, search for GTDB's suffixed
    variants (_A, _B, _A1, etc.) of the same base name and use the first one found."""
    base_name = str(base_name) 
    parsed = lookup_lineage(f"{rank_prefix_char}{base_name}")
    if parsed:
        return parsed, base_name

    candidates = search_gtdb_nodes(base_name)
    suffixed = [c for c in candidates if re.match(rf"^{rank_prefix_char}{re.escape(base_name)}_[A-Z]\d*$", c)]
    for candidate_node in suffixed:
        parsed = lookup_lineage(candidate_node)
        if parsed:
            resolved_name = candidate_node[len(rank_prefix_char):]
            return parsed, resolved_name

    return None, None

## 3.3 Commentary procedence

In [ ]:
records = []
for _, r in genus_queries.iterrows():
    # add a real family-level fallback after the genus attempts fail, so at least family is scriptly verified and retrieved
  
    original_genus = r["Genus"]
    original_family = r["Family"]
    rank_prefix_char = r["GTDB_query_rank"]        # "f__" or "g__" — actually use it
    cleaned_genus = r["GTDB_query_genus"]           # already-stripped name from clean_gtdb_query
    row = {"GID": r["GID"], "Genus": original_genus}

    parsed, resolved_name = lookup_lineage_with_suffix_retry(cleaned_genus, rank_prefix_char)
    if parsed:
        row.update(parsed)
        row["GTDB_status"] = "direct" if resolved_name == cleaned_genus else f"resolved_via_suffix({resolved_name})"
            
    # genus totally failed -- fall back to the ORIGINAL Family value
    if not parsed and pd.notna(original_family):
        fam_parsed, fam_resolved = lookup_lineage_with_suffix_retry(
            str(original_family).strip(), "f__")
            
        if fam_parsed:
            row.update(fam_parsed)
            row["GTDB_status"] = f"resolved_via_family({fam_resolved})"
            parsed = fam_parsed

    if not parsed:
        row["GTDB_status"] = "not_in_gtdb"
    
    records.append(row)  # always runs, every genus gets a row regardless of outcome

table_b = pd.DataFrame(records)

In [ ]:
table_b.sample(10)

,GID,Genus,Kingdom,Phylum,Class,Order,Family,GTDB_status
569,61,Saccharibacteria_unclassified,NaN,NaN,NaN,NaN,NaN,not_in_gtdb
413,709,Tatlockia,Bacteria,Pseudomonadota,Gammaproteobacteria,Legionellales,Legionellaceae,direct
56,150,Prolixibacteraceae,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Prolixibacteraceae,direct
781,559,Pleomorphomonas,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhizobiales,Pleomorphomonadaceae,direct
216,486,Nitrotoga,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Gallionellaceae,direct
545,801,SR-FBR-L83_uncultured,NaN,NaN,NaN,NaN,NaN,not_in_gtdb
627,790,JG34-KF-161_uncultured,NaN,NaN,NaN,NaN,NaN,not_in_gtdb
613,750,Family_XI_unclassified,NaN,NaN,NaN,NaN,NaN,not_in_gtdb
506,806,GIF9_uncultured,NaN,NaN,NaN,NaN,NaN,not_in_gtdb
215,483,Nitrosomonas,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Nitrosomonadaceae,direct


## 4 Merging working GTDB harmonisation Lineage with non harmonisated

In [ ]:
working_GTDB = biotot.merge(table_b, on="GID", how="left", suffixes=("_original", "_gtdb"))

In [ ]:
RANKS_TO_COMPARE = [
    "Kingdom",
    "Phylum",
    "Class",
    "Order",
    "Family"
]

def make_comment(row):

    # No GTDB result
    if row["GTDB_status"] == "not_in_gtdb":
        return "No GTDB lineage retrieved"

    diffs = []

    for rank in RANKS_TO_COMPARE:

        original = row.get(f"{rank}_original")
        gtdb_val = row.get(f"{rank}_gtdb")

        if pd.isna(gtdb_val):
            continue

        if pd.isna(original):
            diffs.append(f"{rank}: missing -> {gtdb_val}")

        elif str(original).strip() != str(gtdb_val).strip():
            diffs.append(f"{rank}: {original} -> {gtdb_val}")

    base_comment = "; ".join(diffs) if diffs else "Exact lineage match"

    if str(row["GTDB_status"]).startswith("resolved_via_family"):
        return f"[genus unresolved, family-level only] {base_comment}"

    return base_comment

working_GTDB["Comment"] = working_GTDB.apply(make_comment, axis=1)
mismatches = working_GTDB[working_GTDB["Comment"].str.startswith(("Phylum", "Class", "Order", "Family"))]

In [ ]:
mismatches.sample(5)

In [ ]:
working_GTDB.sample(5)

,Domain,Phylum_original,Class_original,Order_original,Family_original,Genus_original,GID,sum_all_sites,Genus_gtdb,Kingdom,Phylum_gtdb,Class_gtdb,Order_gtdb,Family_gtdb,GTDB_status,Comment
308,Bacteria,Bacillota,Bacilli,Lactobacillales,Carnobacteriaceae,Carnobacteriaceae_unclassified,67,0.003363,Carnobacteriaceae_unclassified,Bacteria,Bacillota,Bacilli,Lactobacillales,Carnobacteriaceae,direct,Exact lineage match
474,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Bacteroidaceae,Prevotella_7,569,11.714687,Prevotella,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Bacteroidaceae,direct,Exact lineage match
163,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Burkholderiaceae,Acidovorax,17,191.084839,Acidovorax,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Burkholderiaceae_C,direct,Family: Burkholderiaceae -> Burkholderiaceae_C
196,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Burkholderiaceae,Noviherbaspirillum,490,15.146394,Noviherbaspirillum,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Burkholderiaceae_D,direct,Family: Burkholderiaceae -> Burkholderiaceae_D
56,Bacteria,Bacteroidota,Bacteroidota,Bacteroidia,Bacteroidales,Prolixibacteraceae,150,0.015255,Prolixibacteraceae,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Prolixibacteraceae,direct,Class: Bacteroidota -> Bacteroidia; Order: Bac...


## 4.1 Checking the type of no found lineage

In [ ]:
not_found = working_GTDB[working_GTDB["GTDB_status"] == "not_in_gtdb"].copy()

def categorize(genus):
    genus = str(genus)
    if any(m in genus.lower() for m in ("uncultured", "clone", "enrichment_culture", "bacterium_")):
        return "environmental_clone_or_placeholder"
    if genus.startswith("Candidatus_"):
        return "candidatus_name"
    if "_" in genus and re.match(r"^[A-Z][a-z]+_[A-Z][a-z]+$", genus):
        return "composite_two_genus_name"
    if re.match(r"^[A-Z][a-z]+$", genus):
        return "clean_genus_name"  # worth checking individually -- this is the group most likely a real gap or rename
    return "other"

not_found["category"] = not_found["Genus_original"].apply(categorize)
not_found["category"].value_counts()

category
environmental_clone_or_placeholder    65
other                                 56
clean_genus_name                      20
candidatus_name                        6
Name: count, dtype: int64

In [ ]:
# Completeness check: has every row actually been resolved against GTDB
# (genus, or at minimum family), across all the runs so far?

unresolved = table_b[table_b["GTDB_status"] == "not_in_gtdb"]
missing_lineage = table_b[table_b[["Kingdom","Phylum","Class","Order","Family"]].isna().all(axis=1)]

print(f"Total rows: {len(table_b)}")
print(f"Rows still not_in_gtdb: {len(unresolved)}")
print(f"Rows with completely empty lineage: {len(missing_lineage)}")

if len(unresolved) == 0 and len(missing_lineage) == 0:
    print("All rows verified up to at least Family level.")
else:
    print("Rows needing attention:")
    print(unresolved[["GID", "Genus"]])


Second Run
environmental_clone_or_placeholder    66  
other                                 56  
clean_genus_name                      20  
candidatus_name                        6  

The code was run several times and the script was modified accordingly. The missed harmonisation was done manually, for instance Bosea genus was changed to Allobosea which is the new name on GTDB classification, each change was recorder on the column Comment which lives on Biotot, there lives, the original lineage (consolidated) coming from biotot_nocurated which has some partially curated non homogenised lineage, a first run has been assimilated intop and a second run which feeds the first run harmonisation, the runs are then implant direcly on the consolidated columns in "biotot". In summary: 
biotot_noncurated = original data nothing done
biotot = live sheets that get updated depending on the runs and has a permanent column of comment summary of what have happened. Come from biotot_noncurated--> manual harmonisation -->working_gtdb run 1 -->working_gtdb run 2 
Working_gtdb = scratch sheet, one run at a time — gets overwritten each time the notebook runs, doesn't retain history.
biotot_merged = the actual history/archive. It holds, as separate preserved columns:  manual harmonisation (the pre-harmonisation consolidated state), the first run's output, and the second run's output — each copied out before Working_gtdb got overwritten by the next run.

## 4.2 Saving to Excel
The final harmonisation requires to look at the left non harmonised data so that it could be manualy curated

In [ ]:
# Save to excel base_harmonised as a new sheet "working_GTDB" without deleting the other sheets, using openpyxl engine
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    working_GTDB.to_excel(writer, sheet_name="working_GTDB", index=False)

# 5. New data Parcing
Found under revising some files and matching it to the late publication on 2024.

In [112]:
df = pd.read_csv("data/Linage_Family_from_Taxonomietabelle.csv", header=None)

In [113]:
# Combine Row 0 (TubeID) and Row 1 (FF... ID) to keep track of both sample identifiers
header_cols = []
for col in range(df.shape[1]):
    val1 = str(df.iloc[0, col]).strip() if pd.notna(df.iloc[0, col]) else ""
    val2 = str(df.iloc[1, col]).strip() if pd.notna(df.iloc[1, col]) else ""
    if val1 and val2:
        header_cols.append(f"{val1}_{val2}")
    elif val1:
        header_cols.append(val1)
    elif val2:
        header_cols.append(val2)
    else:
        header_cols.append(f"Col_{col}")
# Slice the dataframe to drop the first two header rows, then apply new column names
df_clean = df.iloc[2:].copy()
df_clean.columns = header_cols
df_clean.reset_index(drop=True, inplace=True)

In [114]:
# 3. Split the OTU ID from the Kingdom string in the first column
# "4362609.0 k__Bacteria" -> OTUID: "4362609.0", Kingdom: "k__Bacteria"
split_first_col = df_clean.iloc[:, 0].str.split(" ", n=1, expand=True)
df_clean.insert(0, "OTU_ID", split_first_col[0])
df_clean.iloc[:, 1] = split_first_col[1]  # Overwrite the original column with just the Kingdom string

In [115]:
# Rename taxonomy columns so they are clean and recognizable
tax_ranks = ["Kingdom", "Phylum", "Class", "Order", "Family"]
for i, rank in enumerate(tax_ranks):
    df_clean.rename(columns={df_clean.columns[i + 1]: rank}, inplace=True)
# removing the prefixes "k__", "p__"
tax_columns = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus"]

# Strip the prefixes (e.g., "k__", "p__") only from these columns
for col in tax_columns:
    if col in df_clean.columns:
        # This replaces any letter followed by '__' at the start of the string
        df_clean[col] = df_clean[col].astype(str).str.replace(r'^[a-z]__', '', regex=True)

The lineage from Kingdom to Family was on a page that was separated from the Genus, so off course was difficult to match them because the reorganisation of the lineage by the experts has been continued a long the years, so it was just merged.

In [ ]:
df_genus = pd.read_csv('data/lineage_genus_from_Taxonomietabelle.csv')
# merge family df with genus df
new_data = pd.merge(df_clean, df_genus, on='Family') #, how=
# order the columns Otu, Kingdom, Phylum, Class, Order, Family Genus, rest
new_data = new_data[["OTU_ID", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus"] + [col for col in df_merged.columns if col not in ["OTU_ID", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus"]]] 

In [ ]:
# saving the df to a excel file
new_data.to_excel("data/merged_otus2024.xlsx", index=False)

# 6 Harmonisation New data: Genus -> Family harmonization
Now the new genus from this list are going to be harmonised with the original curated data
Logic :
  1. Match new genera against the baseline (old, ~800-genus) table by EXACT
     genus name first. Anything that matches is done -- no further work.
  2. For genera that don't match directly, try matching after light name
     cleaning (removing _unclassified / _uncultured / _sp_XXX / etc. --
     reuses the same cleaning logic from 1_GTDB.py).
  3. Whatever is still unmatched gets grouped by Family and compared against
     the baseline's Family assignments:
       - If exactly ONE baseline genus shares that Family -> collapse the
         new genus into it (conservative, single-candidate assumption).
       - If MULTIPLE baseline genera share that Family -> flag for manual
         review, list all candidates (do not auto-decide).
       - If NO baseline genus shares that Family -> genuinely new Family,
         keep separate and flag as novel.
  4. Output is one unified table with a Match_Type column so you can filter
     to exactly what needs your manual attention.
## 6.1 Name cleaning (same placeholder-stripping logic used in 1_GTDB.py)

In [ ]:
base_harmonised = pd.read_excel("data/Biotot.xlsx", sheet_name="Biotot")
#new_data = pd.read_excel("data/merged_taxonomy_from_taxonomytabelle.xlsx")

In [ ]:
PLACEHOLDER_TOKEN = (
    r"(?:unclassified|uncultured|unknown|bacterium|archaeon|"
    r"proteobacterium|group|clade)"
)


def clean_genus_name(genus):
    """Strip placeholder suffixes so near-matches can be found.
    Repeats until stable, so stacked suffixes are fully removed."""
    if pd.isna(genus):
        return genus
    genus = str(genus).strip()

    prev = None
    while prev != genus:
        prev = genus
        genus = re.sub(rf"_{PLACEHOLDER_TOKEN}$", "", genus, flags=re.IGNORECASE)

    genus = re.sub(r"_sensu_stricto(_\d+)?$", "", genus, flags=re.IGNORECASE)
    genus = re.sub(r"_sp_[A-Za-z0-9]+$", "", genus)
    genus = re.sub(r"_\d+$", "", genus)

    return genus

In [ ]:
new_data["Genus"] = new_data["Genus"].apply(clean_genus_name)

## 6.2. Core harmonization logic

In [ ]:
def harmonize(new_df, baseline_df,
              genus_col="Genus", family_col="Family", gid_col="GID"):
    """
    new_df:      new/unresolved data. Needs at least Genus + Family columns.
    baseline_df: old 800-genus table. Needs GID + Genus + Family columns.

    Returns a copy of new_df with extra columns:
      Match_Type, Assigned_Genus, Assigned_GID, Family_Candidates
    """
    baseline_by_genus = dict(zip(baseline_df[genus_col], baseline_df[gid_col]))
    baseline_by_family = (
        baseline_df.groupby(family_col)[genus_col].apply(list).to_dict()
    )
    baseline_genus_to_gid = dict(zip(baseline_df[genus_col], baseline_df[gid_col]))

    out_rows = []
    for _, row in new_df.iterrows():
        genus = row[genus_col]
        family = row.get(family_col)

        result = {
            "Match_Type": None,
            "Assigned_Genus": None,
            "Assigned_GID": None,
            "Family_Candidates": None,
        }

        # Step 1: exact genus match
        if genus in baseline_by_genus:
            result["Match_Type"] = "exact_genus_match"
            result["Assigned_Genus"] = genus
            result["Assigned_GID"] = baseline_by_genus[genus]

        else:
            # Step 2: cleaned-name match
            cleaned = clean_genus_name(genus)
            if cleaned != genus and cleaned in baseline_by_genus:
                result["Match_Type"] = f"matched_after_cleaning({cleaned})"
                result["Assigned_Genus"] = cleaned
                result["Assigned_GID"] = baseline_genus_to_gid[cleaned]

            else:
                # Step 3: Family-level fallback
                candidates = baseline_by_family.get(family, [])
                if len(candidates) == 1:
                    result["Match_Type"] = "merged_by_family_single_candidate"
                    result["Assigned_Genus"] = candidates[0]
                    result["Assigned_GID"] = baseline_genus_to_gid[candidates[0]]
                elif len(candidates) > 1:
                    result["Match_Type"] = "family_multiple_candidates_MANUAL_REVIEW"
                    result["Family_Candidates"] = "; ".join(candidates)
                else:
                    result["Match_Type"] = "novel_family_or_genus"

        out_rows.append({**row.to_dict(), **result})

    return pd.DataFrame(out_rows)


In [ ]:
harmonised = harmonize(new_data, base_harmonised,
              genus_col="Genus", family_col="Family", gid_col="GID")

In [ ]:
print(harmonised[["Genus", "Family", "Match_Type", "Assigned_Genus",
                   "Assigned_GID", "Family_Candidates"]].to_string(index=False))

print("\nSummary:")
print(harmonised["Match_Type"].value_counts())

# harmonised.to_excel("data/harmonized_genus_family_matches.xlsx", index=False)

only auto-collapse when the Family is generally sparse (few genera total), and be more cautious even with one candidate if that Family is one of your ecologically important ones — I can add a size/abundance-aware threshold. Just let me know